# B08 — DIME pada fitur mammografi (adaptasi eksploratif)
Mengestimasi KL per kelas dengan objective Donsker–Varadhan, lalu H(Y|Z) dan invers Fano biner. **Z = fitur ResNet50 ImageNet B07**, bukan piksel asli. Implementasi mandiri berdasarkan Zhang et al., *DIME: An Information-Theoretic Difficulty Measure for AI Datasets*, NeurIPS 2020, persamaan 3–5 dan Algorithm 1. https://openreview.net/pdf?id=kvqPFy0hbF

Perbedaan eksplisit dari paper: fitur 2048 dimensi; MLP residual kecil 128 unit (bukan 4096); Adam; cross-fitting pasien dengan inner selection; hanya train benchmark. Hasil bergantung representasi, ukuran sampel, estimator dan optimisasi. **Bukan batas error tersertifikasi atau prediksi akurasi CNN.**

Alur: cache B07 → audit → 3 outer folds pasien → inner fit/selection → dua critic per prior → held-out KL → entropy/Fano. Ulang seed 42–44. Natural memakai prevalensi citra; balanced memakai prior 0,5 melalui pembobotan, bukan membuang citra. Ukuran dataset tetap berbeda. Tidak memakai subset 42 citra B07A untuk melatih estimator karena terlalu kecil.

Jalankan dari atas ke bawah di runtime baru. Cache B07 harus tersedia di Drive. Tidak melatih Teacher/Student; tidak membuka test atau validation benchmark. Output baru dipisahkan dari B07. Run selesai dapat dipakai ulang; run terputus diulang dari awal pada unit dataset–seed–fold–prior.


## Cell 1 — Lingkungan dan Drive

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
from pathlib import Path
import sys, json, hashlib, platform, copy, time, zipfile
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from scipy.optimize import brentq
from scipy.special import xlogy
from IPython.display import display
from google.colab import drive, files
drive.mount('/content/drive')
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:',DEVICE,'PyTorch:',torch.__version__)


## Cell 2 — Konfigurasi tetap sebelum melihat hasil

In [ ]:
ROOT=Path('/content/drive/MyDrive/Research/outputs/benchmark_4datasets')
B07_DIR=ROOT/'b07_pycol_v1/453add0862207787'
OUTPUT_ROOT=ROOT/'b08_dime_features_v1'
DATASETS=['cbis_ddsm','cmmd','inbreast','mias']
SEEDS=[42,43,44]
FOLDS=3
WIDTH=128
MAX_EPOCHS=150
PATIENCE=20
LEARNING_RATE=1e-3
PRIORS=['natural','balanced']
DOWNLOAD_REPORT=True
assert len(set(DATASETS))==len(DATASETS) and len(set(SEEDS))==len(SEEDS)


## Cell 3 — Estimator dan pemeriksaan numerik
Setiap kelas memiliki critic sendiri. KL mentah tetap disimpan meskipun negatif. KL diproyeksikan ke [0, −log(prior kelas)] hanya untuk skor diagnostik bounded; flag koreksi selalu disimpan. Skor mentah yang melanggar batas teoretis tidak diberi nilai valid. Model nol merupakan kandidat awal, dipilih hanya lewat inner selection. Outer holdout tidak dipakai memilih epoch.

In [ ]:
ENGINE_SOURCE="def sha(p):\n    h=hashlib.sha256()\n    with open(p,'rb') as f:\n        for b in iter(lambda:f.read(1048576),b''): h.update(b)\n    return h.hexdigest()\ndef objsha(x): return hashlib.sha256(json.dumps(x,sort_keys=True,allow_nan=False).encode()).hexdigest()\ndef save(p,x):\n    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)\n    tmp=p.with_suffix(p.suffix+'.tmp');tmp.write_text(json.dumps(x,indent=2,allow_nan=False));os.replace(tmp,p)\ndef entropy(p): return float(-np.sum(xlogy(p,p)))\ndef fano(h):\n    if h < -1e-10 or h > np.log(2)+1e-10: return None\n    h=float(np.clip(h,0,np.log(2)))\n    if h==0:return 0.\n    if abs(h-np.log(2))<1e-12:return .5\n    return float(brentq(lambda e:entropy(np.array([e,1-e]))-h,1e-15,.5))\ndef scores(k,p):\n    k=np.array(k,dtype=float);p=np.array(p,dtype=float)\n    upper=-np.log(p); bounded=np.clip(k,0,upper)\n    H=entropy(p); rawH=H-float(p@k); Hb=H-float(p@bounded)\n    valid=bool(np.all(k>=-1e-8) and np.all(k<=upper+1e-8) and -1e-8<=rawH<=H+1e-8)\n    return dict(prior_0=float(p[0]),prior_1=float(p[1]),label_entropy=H,\n        kl_0_raw=float(k[0]),kl_1_raw=float(k[1]),mi_raw=float(p@k),conditional_entropy_raw=rawH,\n        raw_valid=valid,dime_raw=fano(rawH) if valid else None,\n        kl_0_bounded=float(bounded[0]),kl_1_bounded=float(bounded[1]),\n        conditional_entropy_bounded=Hb,dime_bounded=fano(Hb),\n        correction_applied=bool(np.any(abs(k-bounded)>1e-8)))\nclass Critic(nn.Module):\n    def __init__(self,d,w):\n        super().__init__();self.first=nn.Linear(d,w);self.a=nn.Linear(w,w);self.b=nn.Linear(w,w);self.out=nn.Linear(w,1)\n        nn.init.zeros_(self.out.weight);nn.init.zeros_(self.out.bias)\n    def forward(self,x):\n        h=torch.relu(self.first(x));h=torch.relu(h+self.b(torch.relu(self.a(h))))\n        return self.out(h).squeeze(-1)\ndef dv(t,y,c,p):\n    terms=torch.stack([torch.logsumexp(t[y==j],0)-np.log(int((y==j).sum()))+np.log(p[j]) for j in (0,1)])\n    return t[y==c].mean()-torch.logsumexp(terms,0)\ndef inner_split(frame,outer_train,seed):\n    f=frame.iloc[outer_train]\n    splitter=GroupShuffleSplit(n_splits=200,test_size=.25,random_state=seed)\n    for a,b in splitter.split(f,groups=f.patient_id):\n        ia,ib=outer_train[a],outer_train[b]\n        if all(min(np.bincount(frame.label.to_numpy()[i],minlength=2))>=2 for i in (ia,ib)):return ia,ib\n    raise ValueError('Cannot form inner patient-disjoint split with >=2 images/class')\ndef train_critic(X,y,fit,select,hold,p,c,seed,width,epochs,patience,lr,device):\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n    # Invertible scalar feature rescaling determined only from estimator-fit rows.\n    scale=max(float(np.sqrt(np.mean(X[fit].astype(np.float64)**2))),1e-8)\n    z=torch.as_tensor(X/scale,dtype=torch.float32,device=device)\n    yy=torch.as_tensor(y,dtype=torch.long,device=device)\n    model=Critic(X.shape[1],width).to(device);opt=torch.optim.Adam(model.parameters(),lr=lr)\n    best=copy.deepcopy(model.state_dict());best_value=0.;best_epoch=0;wait=0;history=[]\n    for epoch in range(1,epochs+1):\n        model.train();opt.zero_grad();value=dv(model(z[fit]),yy[fit],c,p)\n        if not torch.isfinite(value):raise ValueError('Nonfinite fit DV')\n        (-value).backward();torch.nn.utils.clip_grad_norm_(model.parameters(),5.);opt.step()\n        model.eval()\n        with torch.no_grad():v=float(dv(model(z[select]),yy[select],c,p))\n        if not np.isfinite(v):raise ValueError('Nonfinite selection DV')\n        history.append(dict(epoch=epoch,fit_dv=float(value.detach()),selection_dv=v))\n        if v>best_value+1e-6:best_value=v;best_epoch=epoch;best=copy.deepcopy(model.state_dict());wait=0\n        else:wait+=1\n        if wait>=patience:break\n    model.load_state_dict(best);model.eval()\n    with torch.no_grad():held=float(dv(model(z[hold]),yy[hold],c,p))\n    if not np.isfinite(held):raise ValueError('Nonfinite held-out DV')\n    state={k:v.cpu() for k,v in best.items()}\n    return held,history,state,dict(best_epoch=best_epoch,selection_dv=best_value,epochs_completed=epoch,\n        stopped_early=epoch<epochs,zero_critic_selected=best_epoch==0,scale=scale)\n"
exec(compile(ENGINE_SOURCE,"b08_dime_engine.py","exec"))
assert abs(fano(np.log(2))-.5)<1e-10 and fano(0)==0
assert abs(fano(entropy(np.array([.1,.9])))-.1)<1e-8
assert scores([0,0],[.5,.5])['dime_bounded']==.5
assert scores([np.log(2)]*2,[.5,.5])['dime_bounded']==0
assert not scores([-.1,0],[.5,.5])['raw_valid']
ytest=torch.tensor([0,0,1,1]); ptest=np.array([.5,.5])
t=torch.tensor([2.,2.,-2.,-2.]);a=dv(t,ytest,0,ptest)
assert torch.allclose(a,dv(t+7,ytest,0,ptest),atol=1e-6)
assert abs(float(dv(torch.zeros(4),ytest,0,ptest)))<1e-7
print('Numerical self-tests passed')


## Cell 4 — Audit cache dan rencana pemisahan pasien
Membaca index serta fitur B07, bukan citra atau prediksi test. Semua citra satu pasien tetap berada di satu bagian. Label campuran per pasien dipertahankan. Natural memakai prior empiris seluruh train sebagai target bersama pada fit/selection/holdout; balanced memakai 0,5. Estimasi tetap berbobot citra, bukan rerata pasien.

In [ ]:
source=json.loads((B07_DIR/'protocol.json').read_text())
assert source['split']=='train' and source['features']==2048
assert objsha(source)[:16]==B07_DIR.name
FRAMES={};FEATURES={};provenance={};plans={};membership=[];audit=[]
for ds in DATASETS:
    folder=B07_DIR/'feature_cache'/ds
    done=json.loads((folder/'COMPLETE.json').read_text())
    f=pd.read_csv(folder/'feature_index.csv',dtype={'sample_id':str,'patient_id':str})
    assert not f[['sample_id','patient_id','label']].isna().any().any()
    assert f.sample_id.is_unique and set(f.label)=={0,1}
    f['label']=f.label.astype(int)
    assert done['manifest']['protocol_id']==B07_DIR.name
    assert done['manifest']['weights_sha256']==source['weights_sha256']
    assert done['manifest']['ordered_ids']==f.sample_id.tolist()
    assert sha(folder/'features.npy')==done['features_sha256']
    X=np.load(folder/'features.npy',allow_pickle=False)
    assert X.shape==(len(f),2048) and np.isfinite(X).all()
    FRAMES[ds]=f;FEATURES[ds]=X
    provenance[ds]={'features_sha256':sha(folder/'features.npy'),'index_sha256':sha(folder/'feature_index.csv')}
    for seed in SEEDS:
        splitter=StratifiedGroupKFold(FOLDS,shuffle=True,random_state=seed)
        held=[]
        for fold,(tr,ho) in enumerate(splitter.split(X,f.label,groups=f.patient_id)):
            fi,se=inner_split(f,tr,seed+fold*100)
            assert all(min(np.bincount(f.label.to_numpy()[ix],minlength=2))>=2 for ix in (fi,se,ho))
            groups=[set(f.patient_id.iloc[ix]) for ix in (fi,se,ho)]
            assert not(groups[0]&groups[1] or groups[0]&groups[2] or groups[1]&groups[2])
            plans[ds,seed,fold]=(fi,se,ho);held.extend(ho.tolist())
            for part,ix in zip(['fit','selection','holdout'],[fi,se,ho]):
                membership.append(f.iloc[ix].assign(dataset=ds,seed=seed,fold=fold,partition=part))
                audit.append(dict(dataset=ds,seed=seed,fold=fold,partition=part,images=len(ix),
                    patients=f.patient_id.iloc[ix].nunique(),negative=int((f.label.iloc[ix]==0).sum()),positive=int((f.label.iloc[ix]==1).sum())))
        assert sorted(held)==list(range(len(f)))
MEMBERSHIP=pd.concat(membership,ignore_index=True);AUDIT=pd.DataFrame(audit)
display(AUDIT)
protocol=dict(version='b08_dime_features_v1',source_protocol_sha256=sha(B07_DIR/'protocol.json'),
    datasets=DATASETS,seeds=SEEDS,folds=FOLDS,width=WIDTH,max_epochs=MAX_EPOCHS,patience=PATIENCE,
    learning_rate=LEARNING_RATE,priors=PRIORS,optimizer='Adam',critic='residual MLP; scalar output; zero candidate',
    target='image weighted train distribution',scaling='fit-only scalar RMS',
    membership_sha256=hashlib.sha256(MEMBERSHIP.to_csv(index=False).encode()).hexdigest(),
    engine_sha256=hashlib.sha256(ENGINE_SOURCE.encode()).hexdigest(),provenance=provenance,
    environment=dict(torch=torch.__version__,numpy=np.__version__,python=platform.python_version(),device=DEVICE))
ID=objsha(protocol)[:16];OUT=OUTPUT_ROOT/ID;OUT.mkdir(parents=True,exist_ok=True)
if (OUT/'protocol.json').exists():assert json.loads((OUT/'protocol.json').read_text())==protocol
save(OUT/'protocol.json',protocol);save(OUT/'source_protocol.json',source)
(OUT/'b08_dime_engine.py').write_text(ENGINE_SOURCE)
MEMBERSHIP.to_csv(OUT/'partition_membership.csv',index=False);AUDIT.to_csv(OUT/'partition_audit.csv',index=False)
print('Output:',OUT)


## Cell 5 — Jalankan estimator
Checkpoint tersimpan per critic. Run selesai diverifikasi dengan hash sebelum digunakan kembali. Hanya fold yang selesai utuh dapat dilewati; fold terputus dihitung ulang. Early stopping berlaku untuk estimator DIME, tidak mengubah eksperimen KD.

In [ ]:
rows=[]
for ds in DATASETS:
    f=FRAMES[ds];X=FEATURES[ds];y=f.label.to_numpy()
    for seed in SEEDS:
        for fold in range(FOLDS):
            fi,se,ho=plans[ds,seed,fold]
            for prior in PRIORS:
                folder=OUT/ds/f'seed_{seed}'/f'fold_{fold}'/prior;folder.mkdir(parents=True,exist_ok=True)
                result_path=folder/'result.json'
                if result_path.exists():
                    r=json.loads(result_path.read_text());assert r['protocol_id']==ID
                    for name,h in r['artifacts'].items():assert sha(folder/name)==h
                    rows.append(r['metrics']);continue
                print(ds,seed,'fold',fold,prior,flush=True)
                p=np.bincount(y,minlength=2)/len(y) if prior=='natural' else np.array([.5,.5])
                kl=[];details=[]
                for cls in (0,1):
                    value,hist,state,detail=train_critic(X,y,fi,se,ho,p,cls,seed+fold*100+cls,
                        WIDTH,MAX_EPOCHS,PATIENCE,LEARNING_RATE,DEVICE)
                    kl.append(value);details.append(detail)
                    pd.DataFrame(hist).to_csv(folder/f'class_{cls}_history.csv',index=False)
                    torch.save(state,folder/f'class_{cls}_critic.pt')
                r=dict(dataset=ds,seed=seed,fold=fold,prior=prior,holdout_images=len(ho),**scores(kl,p))
                save(folder/'training_details.json',details)
                artifacts={p.name:sha(p) for p in folder.iterdir() if p.suffix in ('.csv','.pt') or p.name=='training_details.json'}
                save(result_path,dict(protocol_id=ID,metrics=r,artifacts=artifacts))
                rows.append(r)
FOLD_RESULTS=pd.DataFrame(rows);FOLD_RESULTS.to_csv(OUT/'fold_results.csv',index=False)
display(FOLD_RESULTS.round(4))


## Cell 6 — Agregasi dan diagnosis
KL holdout dirata-ratakan antarfold dengan bobot sama, baru entropy/Fano dihitung. SD dihitung atas tiga estimasi per seed, bukan menganggap fold independen. Nilai bounded hanya diagnostik jika ada koreksi. Jika critic nol sering terpilih, skor tinggi dapat berarti estimator gagal belajar; jangan menyimpulkan dataset mustahil dipelajari.

In [ ]:
records=[]
for (ds,seed,prior),g in FOLD_RESULTS.groupby(['dataset','seed','prior']):
    assert len(g)==FOLDS
    p=g[['prior_0','prior_1']].iloc[0].to_numpy(float)
    records.append(dict(dataset=ds,seed=seed,prior=prior,**scores(g[['kl_0_raw','kl_1_raw']].mean().to_numpy(),p),
        corrected_folds=int(g.correction_applied.sum())))
RESULTS=pd.DataFrame(records);RESULTS.to_csv(OUT/'seed_results.csv',index=False)
summary=RESULTS.groupby(['dataset','prior']).agg(dime_bounded_mean=('dime_bounded','mean'),
    dime_bounded_sd=('dime_bounded','std'),valid_raw_seeds=('raw_valid','sum'),
    corrected_folds=('corrected_folds','sum'),seeds=('seed','count')).reset_index()
summary['interpretation']='exploratory representation-dependent estimate; not certified error bound'
summary.to_csv(OUT/'summary.csv',index=False)
display(summary.round(4))
print('Bandingkan natural dengan natural dan balanced dengan balanced. Balanced tidak menyamakan ukuran dataset.')
print('Per-class: KL lebih kecil berarti distribusi kelas lebih mirip marginal; jangan menyebutnya per-class accuracy.')
print('MIAS abnormal-only; INbreast target BI-RADS 1/2 vs 5/6: target label antar-dataset tidak identik.')


## Cell 7 — Ekspor laporan
Kirim ZIP untuk pemeriksaan convergence, koreksi KL, prior dan kestabilan seed. Checkpoint tetap di Drive; laporan menyertakan hasil, trace training, protokol, serta pembagian pasien.

In [ ]:
report_files=[p for p in OUT.rglob('*') if p.is_file() and p.suffix!='.pt' and p.name!='report_manifest.json']
save(OUT/'report_manifest.json',dict(protocol_id=ID,files={str(p.relative_to(OUT)):sha(p) for p in report_files}))
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
REPORT=OUTPUT_ROOT/f'B08_DIME_{ID}_{stamp}_report.zip'
with zipfile.ZipFile(REPORT,'w',zipfile.ZIP_DEFLATED) as z:
    for p in report_files+[OUT/'report_manifest.json']:z.write(p,str(p.relative_to(OUT)))
print(REPORT)
if DOWNLOAD_REPORT:files.download(str(REPORT))
